# Chapter 6: Profile grid underfill on an NVIDIA H100

This notebook profiles the exact final autotuned matrix-multiplication
kernel from Chapter 5. It runs two FP16 products:

- `underfilled`: `(M, N, K) = (128, 256, 32768)`
- `square_4096`: `(M, N, K) = (4096, 4096, 4096)`

Run every cell in order. Upload only this notebook to the H100 system.
The first code cell creates a temporary Python target because NVIDIA
Nsight Compute must start a separate process. The final cell packages
the environment record, correctness results, timings, console logs,
launch records, and `.ncu-rep` reports into
`CH06_H100_profile_results.zip`.

The Nsight cells require the `ncu` command and permission to read GPU
performance counters. If they report `ERR_NVGPUCTRPERM`, keep the text
log and ask the machine administrator to enable counter access.


## Create the profiler target

The following cell contains the six Chapter 5 autotune configurations,
the final grouped tile mapping, the compiler hints, the masked K loop,
and the Chapter 5 launch wrapper. Do not change its block sizes before
collecting the first results.

The script completes autotuning before it calls `cudaProfilerStart`.
Nsight Compute therefore measures one launch that uses the selected
configuration instead of measuring the autotuner's candidate launches.


In [ ]:
%%writefile CH06_h100_profile_target.py

from __future__ import annotations

import argparse
import json
import platform
from pathlib import Path

import torch
import triton
import triton.language as tl


PROFILE_CASES = {
    "underfilled": {"M": 128, "N": 256, "K": 32768},
    "square_4096": {"M": 4096, "N": 4096, "K": 4096},
}


def matmul_autotune_configs():
    return [
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 256, "BLOCK_K": 64, "GROUP_M": 8},
            num_warps=8, num_stages=3,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 256, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
    ]


@triton.autotune(
    configs=matmul_autotune_configs(),
    key=["M", "N", "K"],
)
@triton.jit
def optimized_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_in_group = pid % num_pid_in_group
    pid_m = first_pid_m + pid_in_group % group_size_m
    pid_n = pid_in_group // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_m = tl.max_contiguous(
        tl.multiple_of(offs_m, BLOCK_M), BLOCK_M
    )
    offs_n = tl.max_contiguous(
        tl.multiple_of(offs_n, BLOCK_N), BLOCK_N
    )

    a_ptrs = (
        a_ptr
        + offs_m[:, None] * stride_am
        + offs_k[None, :] * stride_ak
    )
    b_ptrs = (
        b_ptr
        + offs_k[:, None] * stride_bk
        + offs_n[None, :] * stride_bn
    )

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_tile in range(0, tl.cdiv(K, BLOCK_K)):
        valid_k = offs_k < K - k_tile * BLOCK_K
        a_mask = (offs_m[:, None] < M) & valid_k[None, :]
        b_mask = valid_k[:, None] & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = (
        c_ptr
        + offs_m[:, None] * stride_cm
        + offs_n[None, :] * stride_cn
    )
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)


def optimized_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.ndim != 2 or b.ndim != 2:
        raise ValueError("a and b must both be matrices")
    if a.shape[1] != b.shape[0]:
        raise ValueError("the inner matrix dimensions must match")
    if a.device != b.device or a.dtype != b.dtype:
        raise ValueError("a and b must have the same device and dtype")
    if not a.is_cuda or not a.is_contiguous() or not b.is_contiguous():
        raise ValueError("a and b must be contiguous GPU tensors")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("this kernel accepts FP16 or BF16 inputs")

    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"])
        * triton.cdiv(N, meta["BLOCK_N"]),
    )
    optimized_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c


def launch_optimized_matmul(a, b, c):
    M, K = a.shape
    _, N = b.shape
    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"])
        * triton.cdiv(N, meta["BLOCK_N"]),
    )
    optimized_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
    )


def selected_matmul_config():
    config = getattr(optimized_matmul_kernel, "best_config", None)
    if config is None:
        raise RuntimeError("The autotuner has not selected a configuration.")
    values = config.kwargs
    return {
        "BLOCK_M": int(values["BLOCK_M"]),
        "BLOCK_N": int(values["BLOCK_N"]),
        "BLOCK_K": int(values["BLOCK_K"]),
        "GROUP_M": int(values["GROUP_M"]),
        "num_warps": int(config.num_warps),
        "num_stages": int(config.num_stages),
    }


def make_inputs(case_name, seed=0):
    if case_name not in PROFILE_CASES:
        raise ValueError(f"Unknown case: {case_name}")
    shape = PROFILE_CASES[case_name]
    device = torch.device("cuda", torch.cuda.current_device())
    torch.manual_seed(seed)
    a = torch.randn(
        (shape["M"], shape["K"]),
        device=device,
        dtype=torch.float16,
    )
    b = torch.randn(
        (shape["K"], shape["N"]),
        device=device,
        dtype=torch.float16,
    )
    c = torch.empty(
        (shape["M"], shape["N"]),
        device=device,
        dtype=torch.float16,
    )
    return a, b, c


def launch_record(case_name):
    shape = PROFILE_CASES[case_name]
    config = selected_matmul_config()
    device = torch.cuda.current_device()
    properties = torch.cuda.get_device_properties(device)
    program_count = (
        triton.cdiv(shape["M"], config["BLOCK_M"])
        * triton.cdiv(shape["N"], config["BLOCK_N"])
    )
    return {
        "case": case_name,
        **shape,
        "dtype": "float16",
        "selected_config": config,
        "program_count": int(program_count),
        "k_iterations_per_program": int(
            triton.cdiv(shape["K"], config["BLOCK_K"])
        ),
        "streaming_multiprocessors": int(
            properties.multi_processor_count
        ),
        "programs_per_sm": (
            float(program_count) / properties.multi_processor_count
        ),
        "dense_flop_count": int(
            2 * shape["M"] * shape["N"] * shape["K"]
        ),
    }


def collect_environment():
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA-capable NVIDIA GPU is required.")
    device = torch.cuda.current_device()
    properties = torch.cuda.get_device_properties(device)
    capability = torch.cuda.get_device_capability(device)
    try:
        target = triton.runtime.driver.active.get_current_target()
        backend = target.backend
        architecture = str(target.arch)
    except Exception as error:
        backend = "unavailable"
        architecture = f"unavailable: {error}"
    return {
        "gpu": properties.name,
        "streaming_multiprocessors": int(
            properties.multi_processor_count
        ),
        "total_memory_gib": properties.total_memory / 2**30,
        "compute_capability": f"{capability[0]}.{capability[1]}",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "triton": triton.__version__,
        "cuda_runtime": torch.version.cuda,
        "triton_backend": backend,
        "triton_architecture": architecture,
    }


def high_precision_reference(a, b):
    previous_precision = torch.get_float32_matmul_precision()
    try:
        torch.set_float32_matmul_precision("highest")
        reference = torch.matmul(a.float(), b.float())
    finally:
        torch.set_float32_matmul_precision(previous_precision)
    return reference.to(dtype=a.dtype)


def validate_case(case_name):
    a, b, c = make_inputs(case_name)
    launch_optimized_matmul(a, b, c)
    torch.cuda.synchronize()
    expected = high_precision_reference(a, b)
    torch.testing.assert_close(c, expected, rtol=1e-2, atol=1e-2)
    difference = (c.float() - expected.float()).abs()
    record = launch_record(case_name)
    record.update(
        {
            "correctness": "pass",
            "max_abs_error": float(difference.max().item()),
            "mean_abs_error": float(difference.mean().item()),
        }
    )
    return record


def benchmark_case(case_name, warmup_ms=50, measurement_ms=200):
    a, b, c = make_inputs(case_name)

    def run_triton():
        launch_optimized_matmul(a, b, c)

    def run_pytorch():
        torch.matmul(a, b, out=c)

    run_triton()
    torch.cuda.synchronize()
    config = selected_matmul_config()
    triton_ms = float(
        triton.testing.do_bench(
            run_triton,
            warmup=warmup_ms,
            rep=measurement_ms,
        )
    )
    pytorch_ms = float(
        triton.testing.do_bench(
            run_pytorch,
            warmup=warmup_ms,
            rep=measurement_ms,
        )
    )
    record = launch_record(case_name)
    record["selected_config"] = config
    operation_count = record["dense_flop_count"]
    record.update(
        {
            "triton_median_ms": triton_ms,
            "pytorch_median_ms": pytorch_ms,
            "triton_tflops": operation_count / triton_ms * 1e-9,
            "pytorch_tflops": operation_count / pytorch_ms * 1e-9,
            "triton_over_pytorch": pytorch_ms / triton_ms,
        }
    )
    return record


def profile_case(case_name, results_directory):
    a, b, c = make_inputs(case_name)

    # The first launch completes autotuning before profiler collection starts.
    launch_optimized_matmul(a, b, c)
    torch.cuda.synchronize()
    record = launch_record(case_name)
    output_path = Path(results_directory)
    output_path.mkdir(parents=True, exist_ok=True)
    with (output_path / f"{case_name}_launch.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(record, handle, indent=2)

    range_name = f"chapter6_{case_name}"
    torch.cuda.cudart().cudaProfilerStart()
    torch.cuda.nvtx.range_push(range_name)
    try:
        launch_optimized_matmul(a, b, c)
        torch.cuda.synchronize()
    finally:
        torch.cuda.nvtx.range_pop()
        torch.cuda.cudart().cudaProfilerStop()

    print(json.dumps(record, indent=2))


def parse_args():
    parser = argparse.ArgumentParser(
        description="Profile the exact final Chapter 5 Triton matmul kernel."
    )
    parser.add_argument(
        "--case",
        choices=sorted(PROFILE_CASES),
        required=True,
    )
    parser.add_argument("--profile", action="store_true")
    parser.add_argument(
        "--results-directory",
        default="ch06_h100_results",
    )
    return parser.parse_args()


def main():
    args = parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA-capable NVIDIA GPU is required.")
    print(json.dumps(collect_environment(), indent=2))
    if args.profile:
        profile_case(args.case, args.results_directory)
    else:
        print(json.dumps(validate_case(args.case), indent=2))


if __name__ == "__main__":
    main()


## Record the H100 environment

This cell verifies CUDA access and records the exact GPU variant, SM
count, compute capability, driver, CUDA runtime, PyTorch, Triton, and
Nsight Compute versions. Keep this output with the profiler reports.


In [ ]:
import importlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import CH06_h100_profile_target as profile_target

profile_target = importlib.reload(profile_target)
RESULTS_DIRECTORY = Path("ch06_h100_results")
RESULTS_DIRECTORY.mkdir(exist_ok=True)

environment = profile_target.collect_environment()

def capture_command(command):
    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return {
        "command": command,
        "returncode": completed.returncode,
        "output": completed.stdout,
    }

nvidia_smi = capture_command(
    [
        "nvidia-smi",
        "--query-gpu=name,uuid,driver_version,memory.total",
        "--format=csv",
    ]
)
ncu_path = shutil.which("ncu")
ncu_version = (
    capture_command([ncu_path, "--version"])
    if ncu_path
    else {
        "command": ["ncu", "--version"],
        "returncode": 127,
        "output": "ncu was not found on PATH",
    }
)
environment["nvidia_smi"] = nvidia_smi
environment["ncu"] = ncu_version

with (RESULTS_DIRECTORY / "environment.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(environment, handle, indent=2)

print(json.dumps(environment, indent=2))
if "H100" not in environment["gpu"]:
    print(
        "WARNING: This notebook was prepared for a full NVIDIA H100; "
        f"the active device reports {environment['gpu']!r}."
    )
if environment["streaming_multiprocessors"] < 100:
    print(
        "WARNING: The active device exposes fewer than 100 SMs. "
        "It may be a MIG partition rather than a full H100."
    )


## Check correctness and reveal the selected configurations

Each first launch runs the Chapter 5 autotuner for that shape. The
result records the selected tile, launch grid, K-loop count, actual SM
count, and numerical error against a high-precision PyTorch reference.
The narrow case may select one, two, four, or eight programs; the
notebook reports what the H100 chooses instead of assuming two.


In [ ]:
PROFILE_CASES = ["underfilled", "square_4096"]
correctness_results = []

for case_name in PROFILE_CASES:
    print(f"Running correctness check: {case_name}")
    result = profile_target.validate_case(case_name)
    correctness_results.append(result)
    print(json.dumps(result, indent=2))

with (RESULTS_DIRECTORY / "correctness_results.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(correctness_results, handle, indent=2)


## Measure ordinary execution outside Nsight Compute

Nsight Compute can replay a kernel while collecting counters, so its
duration is not a benchmark. This cell times the optimized Triton
kernel and `torch.matmul` in an ordinary process after autotuning.


In [ ]:
benchmark_results = []

for case_name in PROFILE_CASES:
    print(f"Benchmarking: {case_name}")
    result = profile_target.benchmark_case(
        case_name,
        warmup_ms=50,
        measurement_ms=200,
    )
    benchmark_results.append(result)
    print(json.dumps(result, indent=2))

with (RESULTS_DIRECTORY / "benchmark_results.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(benchmark_results, handle, indent=2)


## Collect Nsight Compute reports

This cell starts one fresh process for each shape. The target performs
allocation, autotuning, and warm-up before profiler collection starts,
then Nsight Compute records one steady-state Triton launch. The
`LaunchStats` section supplies the grid size and waves per SM. The
`SpeedOfLight` section supplies high-level compute and memory activity.

If `ncu` is missing, the cell keeps the correctness and benchmark
results and prints the exact condition that blocked profiling.


In [ ]:
PROFILE_TARGET = Path("CH06_h100_profile_target.py").resolve()
profiler_runs = []

if ncu_path is None:
    print("Skipping Nsight Compute because ncu was not found on PATH.")
else:
    for case_name in PROFILE_CASES:
        report_base = RESULTS_DIRECTORY / f"ch06_{case_name}"
        command = [
            ncu_path,
            "--profile-from-start",
            "off",
            "--target-processes",
            "all",
            "--section",
            "LaunchStats",
            "--section",
            "SpeedOfLight",
            "--page",
            "details",
            "--print-details",
            "all",
            "--force-overwrite",
            "--export",
            str(report_base),
            sys.executable,
            str(PROFILE_TARGET),
            "--case",
            case_name,
            "--profile",
            "--results-directory",
            str(RESULTS_DIRECTORY),
        ]
        print("Running:", subprocess.list2cmdline(command))
        completed = subprocess.run(
            command,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )
        log_path = RESULTS_DIRECTORY / f"ch06_{case_name}_ncu.txt"
        log_path.write_text(completed.stdout, encoding="utf-8")
        run_record = {
            "case": case_name,
            "command": command,
            "returncode": completed.returncode,
            "log": str(log_path),
        }
        profiler_runs.append(run_record)
        print(completed.stdout)
        if completed.returncode != 0:
            print(
                f"Nsight Compute failed for {case_name}; "
                f"keep {log_path} and inspect the error before retrying."
            )

with (RESULTS_DIRECTORY / "profiler_runs.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(profiler_runs, handle, indent=2)


## Package the results

Run this cell even if Nsight Compute failed. The ZIP will still contain
the environment, correctness, timing, command, and error records needed
to diagnose the run.


In [ ]:
archive_path = Path(
    shutil.make_archive(
        "CH06_H100_profile_results",
        "zip",
        root_dir=RESULTS_DIRECTORY,
    )
).resolve()

print(f"Created: {archive_path}")
print("Files included:")
for path in sorted(RESULTS_DIRECTORY.iterdir()):
    print(f"  {path.name} ({path.stat().st_size} bytes)")
print("\nSend back CH06_H100_profile_results.zip.")
